[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/4-context-engineering.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 4. Context engineering

The loop from tutorial 3 will read whatever you stuff into the prompt. Stuffing is how a Tuesday huddle turns into context rot.

This notebook adds a note catalog. Titles go in the system prompt. Bodies come back through `read_note`. One note is over the cap so you can see the refusal. The picnic note is long and useless for a carton count. The run should not load it.

The shelf tool is imported, so the oat-milk count is checked against the database as well as the note.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, `setup_colab` in `tutorial/common/colab.py` reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `4-context-engineering.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '4-context-engineering'
LESSON_NUMBER = 4
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import reset_db


## Tutorial 4 — notes under a budget

The prompt gets titles. `read_note` returns one short body. The picnic note is over the cap on purpose.

In [ ]:
from tutorial.common.harness import DATA, register

NOTES = DATA / "notes"
PER_NOTE_CAP = 500
TURN_BUDGET = 600


def _note_path(name):
    if not isinstance(name, str) or not name.strip():
        return "ERROR: name is required. Example: oat-milk.md"
    cleaned = name.strip().replace("\\", "/")
    if "/" in cleaned or cleaned.startswith("."):
        return "ERROR: name must be a file in the notes catalog, not a path."
    if not cleaned.endswith(".md"):
        return "ERROR: notes are markdown files, such as oat-milk.md."
    candidate = (NOTES / cleaned).resolve()
    if candidate.parent != NOTES.resolve():
        return "ERROR: name escapes the notes folder."
    if not candidate.is_file():
        return "ERROR: " + cleaned + " is not in the catalog."
    return candidate


def _note_title(text):
    for line in text.splitlines():
        if line.startswith("# "):
            return line[2:].strip()
    return "(untitled)"


def list_notes(args):
    """Titles and sizes only. Bodies stay on disk until read_note."""
    del args
    lines = ["NOTES"]
    for path in sorted(NOTES.glob("*.md")):
        text = path.read_text(encoding="utf-8")
        lines.append(
            "- " + path.name + " | " + _note_title(text) + " | " + str(len(text)) + " chars"
        )
    return "\n".join(lines)


def read_note(args):
    found = _note_path(args.get("name", ""))
    if isinstance(found, str):
        return found
    text = found.read_text(encoding="utf-8")
    if len(text) > PER_NOTE_CAP:
        return (
            "ERROR: " + found.name + " is " + str(len(text))
            + " characters. The per-note cap is " + str(PER_NOTE_CAP)
            + ". File a shorter note."
        )
    return "NOTE: " + found.name + "\n\n" + text


def paste_chars():
    total = 0
    for path in NOTES.glob("*.md"):
        total += len(path.read_text(encoding="utf-8"))
    return total


register(
    "list_notes",
    "List huddle notes by name, title, and size. Does not return bodies.",
    {},
    [],
    list_notes,
)
register(
    "read_note",
    "Read one short note by file name, such as oat-milk.md. "
    "A note over the per-note cap returns ERROR. The picnic note is an example.",
    {"name": {"type": "string", "description": "File name from list_notes."}},
    ["name"],
    read_note,
)


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
catalog = list_notes({})
print("paste_all_chars:", paste_chars())
print("catalog_chars:", len(catalog))
print("TURN_BUDGET:", TURN_BUDGET)
print("PER_NOTE_CAP:", PER_NOTE_CAP)
print(catalog)
print()
print("oversized note:")
print(read_note({"name": "picnic.md"}))
print()
check(paste_chars() > TURN_BUDGET, "pasting every note exceeds the turn budget")
check(len(catalog) < TURN_BUDGET, "the title catalog fits in the budget")

question = (
    "From the huddle notes, how many oat milk cartons were on hand, "
    "and how many cardamom buns did we bake on Thursday? Check the shelf for OM-32. "
    "Do not load the picnic note."
)
result = run_agent(question, system_text())
print("ANSWER:", result["text"])
joined = "\n".join(result["tool_log"])
check(result["stopped"] == "final", "context run finished")
check("oat-milk.md" in joined and "thursday-buns.md" in joined, "the relevant notes were read")
check("picnic.md" not in joined, "the picnic note was not loaded into the loop")
check("3" in result["text"] and "24" in result["text"], "both counts are in the answer")
check("query_inventory" in joined, "the shelf tool from tutorial 2 still runs")
